# UGATIT - Entraînement sur Modal (B200)

Copie adaptée du notebook UGATIT pour lancer l'entraînement sur Modal avec un workflow reproductible.


## 1. Pré-requis Modal

- Installer Modal localement : `pip install -U modal`
- Authentifier la session : `modal setup`
- Vérifier que tu as accès GPU (`B200`) dans ton workspace Modal.


In [6]:
from pathlib import Path
import os
import math
import shlex
import subprocess

import yaml

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
PROJECT_ROOT


PosixPath('/home/johanu/Downloads/cours_uqac/8INF887 - Apprentissage profond - Hiver 2026 - 01/final-project')

## 2. Construire une config Modal

Deux profils sont inclus:

- `b200_balanced` (recommandé) : plus prudent sur overfitting pour Selfie2Anime.
- `b200_user_spec` : proche de ta proposition initiale (adaptée au code qui entraîne en `epochs`, pas en `iterations`).


In [7]:
MODAL_PROFILE = 'b200_balanced'  # 'b200_balanced' ou 'b200_user_spec'
BASE_CONFIG_PATH = PROJECT_ROOT / 'configs' / 'ugatit_selfie2anime.yaml'
MODAL_CONFIG_PATH = PROJECT_ROOT / 'configs' / 'ugatit_selfie2anime_modal_b200.yaml'

PROFILES = {
    'b200_balanced': {
        'image_size': 512,
        'batch_size': 8,
        'lr': 8e-5,
        'epochs': 260,
        'decay_start_epoch': 130,
        'weight_decay': 1e-4,
        'lambda_cycle': 12.0,
        'lambda_identity': 8.0,
        'lambda_cam': 800.0,
        'random_crop': True,
        'color_jitter': 0.05,
        'num_blocks': 6,
        'run_tag': 'ugatit_modal_b200_balanced_v1',
    },
    'b200_user_spec': {
        'image_size': 512,
        'batch_size': 16,
        'lr': 1e-4,
        'epochs': 320,
        'decay_start_epoch': 160,
        'weight_decay': 1e-4,
        'lambda_cycle': 15.0,
        'lambda_identity': 10.0,
        'lambda_cam': 1200.0,
        'random_crop': True,
        'color_jitter': 0.05,
        'num_blocks': 6,
        'run_tag': 'ugatit_modal_b200_user_spec_v1',
    },
}

if MODAL_PROFILE not in PROFILES:
    raise ValueError(f"Profil inconnu: {MODAL_PROFILE}. Choisir parmi: {list(PROFILES)}")

profile = PROFILES[MODAL_PROFILE]

with open(BASE_CONFIG_PATH, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

config.setdefault('data', {})
config.setdefault('train', {})
config.setdefault('runtime', {})
config.setdefault('model', {}).setdefault('ugatit', {})
config['data'].setdefault('augment', {})

config['data']['root'] = 'data/selfie2anime/unpaired'
config['data']['image_size'] = profile['image_size']
config['data']['batch_size'] = profile['batch_size']
config['data']['augment']['random_horizontal_flip'] = True
config['data']['augment']['color_jitter'] = profile['color_jitter']
config['data']['augment']['random_crop'] = profile['random_crop']

config['model']['ugatit']['num_blocks'] = profile['num_blocks']

config['train']['lr'] = profile['lr']
config['train']['epochs'] = profile['epochs']
config['train']['decay_start_epoch'] = profile['decay_start_epoch']
config['train']['weight_decay'] = profile['weight_decay']
config['train']['lambda_cycle'] = profile['lambda_cycle']
config['train']['lambda_identity'] = profile['lambda_identity']
config['train']['lambda_cam'] = profile['lambda_cam']
config['train']['sample_interval'] = 10
config['train']['mixed_precision'] = True

run_tag = profile['run_tag']
config['runtime']['output_dir'] = f'outputs/{run_tag}'
config['runtime']['checkpoint_dir'] = f'checkpoints/{run_tag}'
config['runtime']['resume_checkpoint'] = None

# Estimation du nombre d'itérations avec la taille locale du dataset.
img_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
train_a = PROJECT_ROOT / 'data' / 'selfie2anime' / 'unpaired' / 'trainA'
train_b = PROJECT_ROOT / 'data' / 'selfie2anime' / 'unpaired' / 'trainB'
count_a = len([p for p in train_a.iterdir() if p.suffix.lower() in img_exts]) if train_a.exists() else 0
count_b = len([p for p in train_b.iterdir() if p.suffix.lower() in img_exts]) if train_b.exists() else 0
steps_per_epoch = math.ceil(max(count_a, count_b) / max(1, profile['batch_size'])) if max(count_a, count_b) > 0 else 0
estimated_iterations = steps_per_epoch * profile['epochs']

with open(MODAL_CONFIG_PATH, 'w', encoding='utf-8') as f:
    yaml.safe_dump(config, f, sort_keys=False, allow_unicode=True)

print(f"Profil: {MODAL_PROFILE}")
print(f"Config écrite: {MODAL_CONFIG_PATH}")
print(f"TrainA={count_a}, TrainB={count_b}, steps/epoch≈{steps_per_epoch}, iterations estimées≈{estimated_iterations}")
config


Profil: b200_balanced
Config écrite: /home/johanu/Downloads/cours_uqac/8INF887 - Apprentissage profond - Hiver 2026 - 01/final-project/configs/ugatit_selfie2anime_modal_b200.yaml
TrainA=3400, TrainB=3400, steps/epoch≈425, iterations estimées≈110500


{'experiment_name': 'ugatit_selfie2anime',
 'model_name': 'ugatit',
 'mode': 'ugatit',
 'seed': 42,
 'data': {'root': 'data/selfie2anime/unpaired',
  'image_size': 512,
  'num_workers': 2,
  'batch_size': 8,
  'augment': {'random_horizontal_flip': True,
   'color_jitter': 0.05,
   'random_crop': True},
  'unpaired_layout': {'train_a': 'trainA',
   'train_b': 'trainB',
   'test_a': 'testA',
   'test_b': 'testB'}},
 'train': {'epochs': 260,
  'decay_start_epoch': 130,
  'lr': 8e-05,
  'beta1': 0.5,
  'beta2': 0.999,
  'lambda_cycle': 12.0,
  'lambda_identity': 8.0,
  'lambda_cam': 800.0,
  'sample_interval': 10,
  'mixed_precision': True,
  'weight_decay': 0.0001},
 'runtime': {'output_dir': 'outputs/ugatit_modal_b200_balanced_v1',
  'checkpoint_dir': 'checkpoints/ugatit_modal_b200_balanced_v1',
  'resume_checkpoint': None},
 'model': {'ugatit': {'num_blocks': 6}}}

## 3. Synchroniser projet + dataset dans un Volume Modal

Ce bloc envoie le code et les données vers un volume persistant (`/project/...`) utilisé par le job remote.

Conseil: le premier upload peut prendre du temps; les suivants sont plus rapides.


In [8]:
MODAL_VOLUME_NAME = 'anime-i2i-selfie2anime'

# Création idempotente du volume via SDK.
import modal
modal.Volume.objects.create(MODAL_VOLUME_NAME, allow_existing=True)


def run_cmd(parts):
    print('$', ' '.join(shlex.quote(p) for p in parts))
    subprocess.run(parts, check=True, cwd=str(PROJECT_ROOT))

In [9]:
# Sync code / config / dépendances / données
run_cmd(['modal', 'volume', 'put', '-f', MODAL_VOLUME_NAME, 'src', '/project/'])
run_cmd(['modal', 'volume', 'put', '-f', MODAL_VOLUME_NAME, 'configs', '/project/'])
run_cmd(['modal', 'volume', 'put', '-f', MODAL_VOLUME_NAME, 'requirements.txt', '/project/requirements.txt'])
run_cmd(['modal', 'volume', 'put', '-f', MODAL_VOLUME_NAME, 'data/selfie2anime/unpaired', '/project/data/selfie2anime/'])


$ modal volume put -f anime-i2i-selfie2anime src /project/
⠋ Uploading file(s) to volume...
⠸ Uploading file(s) to volume...0 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ .
Uploading file(s) to volume... 0:00:00 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ .
                         /project/src/anime_gan_i2i/prepare_data.py 0.0% • • • …
                                /project/src/anime_gan_i2i/train.py 0.0% • • • …
                                /project/src/anime_gan_i2i/infer.py 0.0% • • • …
                               /project/src/anime_gan_i2i/config.py 0.0% • • • …
                               /project/src/anime_gan_i2i/losses.py 0.0% • • • …
                             /project/src/anime_gan_i2i/__init__.py 0.0% • • • …
                              /project/src/anime_gan_i2i/metrics.py 0.0% • • • …
                               /project/src/anime_gan_i2i/models.py 0.0% • • • …
                                 /project/src/anime_gan_i2i/data.py 0.0% • • • …
       /project/src/an

## 4. Lancer l'entraînement UGATIT sur Modal (GPU B200)

Le script Modal utilisé est `scripts/modal_train_ugatit.py`.


In [ ]:
ENV = os.environ.copy()
ENV['UGATIT_MODAL_VOLUME'] = MODAL_VOLUME_NAME
ENV.setdefault('UGATIT_MODAL_GPU', 'H100')  # 'H100' (recommande) ou 'B200'
ENV.setdefault('MODAL_FORCE_BUILD', '1')  # force rebuild pour eviter image stale

cmd = [
    'modal', 'run', 'scripts/modal_train_ugatit.py',
    '--config-rel-path', 'configs/ugatit_selfie2anime_modal_b200.yaml',
]

print('$', ' '.join(shlex.quote(p) for p in cmd))
print('GPU:', ENV['UGATIT_MODAL_GPU'], '| MODAL_FORCE_BUILD:', ENV['MODAL_FORCE_BUILD'])
subprocess.run(cmd, check=True, cwd=str(PROJECT_ROOT), env=ENV)


$ modal run scripts/modal_train_ugatit.py --config-rel-path configs/ugatit_selfie2anime_modal_b200.yaml
GPU: H100 | MODAL_FORCE_BUILD: 1


## 5. Récupérer les artefacts (checkpoints + outputs) en local

On rapatrie le dossier du run pour analyse locale (`history.csv`, `samples`, checkpoints).


In [ ]:
run_tag = PROFILES[MODAL_PROFILE]['run_tag']

run_cmd(['modal', 'volume', 'get', MODAL_VOLUME_NAME, f'/project/outputs/{run_tag}', str(PROJECT_ROOT / 'outputs' / run_tag)])
run_cmd(['modal', 'volume', 'get', MODAL_VOLUME_NAME, f'/project/checkpoints/{run_tag}', str(PROJECT_ROOT / 'checkpoints' / run_tag)])

print('Artefacts récupérés localement.')


$ modal volume get anime-i2i-selfie2anime /project/outputs/ugatit_modal_b200_balanced_v1 '/home/johanu/Downloads/cours_uqac/8INF887 - Apprentissage profond - Hiver 2026 - 01/final-project/outputs/ugatit_modal_b200_balanced_v1'
⠋ Downloading file(s) to local...
⠸ Downloading file(s) to local...0 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ .
⠦ Downloading file(s) to local...0 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ .
⠏ Downloading file(s) to local...0 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ .
⠸ Downloading file(s) to local...0 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ .
⠸ Downloading file(s) to local...poch_010.png 0.0% • • • …
project/outputs/ugatit_modal_b200_balanced_v1/samples/epoch_010.png 0.0% • • • …
project/outputs/ugatit_modal_b200_balanced_v1/samples/epoch_001.png 0.0% • • • …


╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ [Errno 21] Is a directory: '/home/johanu/Downloads/cours_uqac/8INF887 -      │
│ Apprentissage profond - Hiver 2026 -                                         │
│ 01/final-project/outputs/ugatit_modal_b200_balanced_v1'                      │
╰──────────────────────────────────────────────────────────────────────────────╯


CalledProcessError: Command '['modal', 'volume', 'get', 'anime-i2i-selfie2anime', '/project/outputs/ugatit_modal_b200_balanced_v1', '/home/johanu/Downloads/cours_uqac/8INF887 - Apprentissage profond - Hiver 2026 - 01/final-project/outputs/ugatit_modal_b200_balanced_v1']' returned non-zero exit status 1.

## 6. Charger l'historique du run téléchargé


In [ ]:
import pandas as pd

run_tag = PROFILES[MODAL_PROFILE]['run_tag']
history_path = PROJECT_ROOT / 'outputs' / run_tag / 'history.csv'

if history_path.exists():
    history_df = pd.read_csv(history_path)
    display(history_df.tail())
else:
    print(f'Historique introuvable: {history_path}')
